# 10 — E10 Memoria asociativa biológica (TTA-only, ImageNet-C)

Operacionaliza tres mecanismos biológicos sobre un checkpoint DeMemte E6 entrenado en ImageNet limpio, sin reentrenar:

1. **Recuperación asociativa** del codebook como Modern Hopfield.
2. **Pattern completion** iterativo en `z_pool` con gate de familiaridad / unfamiliaridad.
3. **Doble vía CLS**: codebook semántico + buffer episódico EMA.

La integración es una **mezcla suave en `zq_pool`** con `λ_max ≤ 0.1` para preservar la calibración del clasificador downstream.

**Phase 0** corre pre-flight checks sobre ImageNet limpio y condiciones ImageNet-C reales. Si una base falla el clean floor, se omite del ranking final.

Base: `dememte_imagenet_resnet50_vqsa_best.pt` en `experiments/imagenet_dememte/out`.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'dememte').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('repo root:', ROOT)


repo root: /home/nakato/projects/Dememte


In [2]:
import math
import json

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from dememte.config import E6Config
from dememte.data import build_imagenet_c_loader, build_imagenet_loaders, seed_everything
from dememte.evaluation import (
    MEMORY_DIAG_KEYS,
    TEACHER_DIAG_KEYS,
    VQSA_KEYS,
    evaluate_dememte,
    evaluate_dememte_tta,
)
from dememte.io import ensure_dir, load_checkpoint, write_csv, write_json
from dememte.memory import (
    EpisodicBuffer,
    HippocampalConfig,
    HippocampalMemoryAdapter,
    associative_recall,
    effective_codebook,
    familiarity_gate,
)
from dememte.models import make_dememte_variant

BASES = ['dememte_imagenet_resnet50_vqsa']

DATA_ROOT_CLEAN = ROOT / 'experiments' / 'data' / 'imagenet-clean-5k'
DATA_ROOT_C = ROOT / 'experiments' / 'data' / 'imagenet-c-subset'
OUT = ensure_dir(ROOT / 'notebooks' / '10_memory_hippocampal' / 'out')
IMAGENET_OUT = ROOT / 'experiments' / 'imagenet_dememte' / 'out'
DEMEMTE_CHECKPOINT = IMAGENET_OUT / 'dememte_imagenet_resnet50_vqsa_best.pt'
TRAIN_CONFIG_PATH = IMAGENET_OUT / 'train_config.json'

CORRUPTIONS = ['gaussian_noise', 'motion_blur', 'pixelate', 'jpeg_compression']
SEVERITIES = [3, 5]
MAX_SAMPLES_PER_CLASS = None
BATCH_SIZE = 64
NUM_WORKERS = 4

# E10 hyperparameters — defaults grounded in the plan's literature table.
LAMBDA_MAX = 0.1     # Lim 2023 TTN typical α-mix
TAU = 1.0            # match vq_temperature (Ramsauer 2021 controls capacity)
TAU_EPI = 1.0
BETA_SEM = 1.0       # pure semantic
BETA_MIX = 0.5       # semantic + episodic mix
ALPHA_W = 0.1        # Sun-Saxe-Fitzgerald 2023 fast plasticity
ALPHA_S = 0.001      # Spens & Burgess 2024 slow consolidation
EPI_SIZE = 256       # Chandra 2023 capacity scaling

CLEAN_FLOOR_TOL = 0.005   # Wang 2022 CoTTA / Song 2023 EcoTTA forgetting threshold

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
print('bases:', BASES)
print('imagenet clean:', DATA_ROOT_CLEAN)
print('imagenet-c:', DATA_ROOT_C)


device: cuda
bases: ['dememte_imagenet_resnet50_vqsa']
imagenet clean: /home/nakato/projects/Dememte/experiments/data/imagenet-clean-5k
imagenet-c: /home/nakato/projects/Dememte/experiments/data/imagenet-c-subset


## Data


In [3]:
# Build ImageNet clean loaders for source clean accuracy and Phase 0.
# ImageNet-C corruptions are evaluated with real condition loaders, not synthetic transforms.
def load_imagenet_cfg():
    if TRAIN_CONFIG_PATH.exists():
        payload = json.loads(TRAIN_CONFIG_PATH.read_text(encoding='utf-8'))
        cfg = E6Config(**payload['config'])
    else:
        cfg = E6Config(
            dataset='imagenet',
            data_dir=str(DATA_ROOT_CLEAN),
            num_classes=1000,
            backbone_name='resnet50',
            backbone_out_channels=2048,
            quantizer_type='ema_vq',
            vq_kmeans_init=False,
            dead_code_restart=False,
        )
    cfg.dataset = 'imagenet_c'
    cfg.data_dir = str(DATA_ROOT_C)
    cfg.num_classes = 1000
    cfg.batch_size = BATCH_SIZE
    cfg.num_workers = NUM_WORKERS
    cfg.device = device
    return cfg

base_cfg = load_imagenet_cfg()
seed_everything(base_cfg.seed)

clean_train_loader, clean_val_loader, clean_meta = build_imagenet_loaders(
    DATA_ROOT_CLEAN,
    batch_size=base_cfg.batch_size,
    num_workers=base_cfg.num_workers,
    seed=base_cfg.seed,
)
meta = {
    'dataset': 'imagenet_c',
    'clean_meta': clean_meta,
    'imagenet_c_root': str(DATA_ROOT_C),
    'corruptions': CORRUPTIONS,
    'severities': SEVERITIES,
    'max_samples_per_class': MAX_SAMPLES_PER_CLASS,
}
print(meta)


{'dataset': 'imagenet_c', 'clean_meta': {'dataset': 'imagenet', 'root': '/home/nakato/projects/Dememte/experiments/data/imagenet-clean-5k', 'train_size': 5000, 'val_size': 1000, 'num_classes': 1000}, 'imagenet_c_root': '/home/nakato/projects/Dememte/experiments/data/imagenet-c-subset', 'corruptions': ['gaussian_noise', 'motion_blur', 'pixelate', 'jpeg_compression'], 'severities': [3, 5], 'max_samples_per_class': None}


## Loaders y constructores por base


In [4]:
def cfg_for_base(base):
    if base not in BASES:
        raise KeyError(f'Unknown ImageNet base: {base!r}')
    return base_cfg


def ckpt_for_base(base):
    if base not in BASES:
        raise KeyError(f'Unknown ImageNet base: {base!r}')
    return DEMEMTE_CHECKPOINT


def load_base_model(base):
    cfg = cfg_for_base(base)
    model = make_dememte_variant(cfg, device=device)
    load_checkpoint(model, ckpt_for_base(base), device=device, strict=True)
    model.eval()
    model.requires_grad_(False)
    return model


def imagenet_c_loader(corruption, severity, *, max_samples_per_class=MAX_SAMPLES_PER_CLASS):
    return build_imagenet_c_loader(
        DATA_ROOT_C,
        corruption,
        severity,
        batch_size=base_cfg.batch_size,
        num_workers=base_cfg.num_workers,
        max_samples_per_class=max_samples_per_class,
        seed=base_cfg.seed,
    )


def scalar_values(record):
    return {k: v for k, v in record.items() if isinstance(v, (int, float, bool, str, np.floating))}


def summarize_imagenet_c_records(clean_record, corruption_records):
    acc_by_corr = {
        f'corrupt_acc_{corr}': float(np.mean([r['acc'] for r in records]))
        for corr, records in corruption_records.items()
    }
    all_corrupt = [r for records in corruption_records.values() for r in records]
    metrics = {
        'clean_acc': clean_record['acc'],
        'corrupt_acc_avg': float(np.mean(list(acc_by_corr.values()))) if acc_by_corr else 0.0,
        **acc_by_corr,
        'ece_clean': clean_record.get('ece', 0.0),
        'ece_corrupt_avg': float(np.mean([r.get('ece', 0.0) for r in all_corrupt])) if all_corrupt else 0.0,
        'nll_clean': clean_record.get('nll', 0.0),
        'nll_corrupt_avg': float(np.mean([r.get('nll', 0.0) for r in all_corrupt])) if all_corrupt else 0.0,
        'brier_clean': clean_record.get('brier', 0.0),
        'brier_corrupt_avg': float(np.mean([r.get('brier', 0.0) for r in all_corrupt])) if all_corrupt else 0.0,
        'corruption_records': corruption_records,
        'clean_record': clean_record,
    }
    for key in VQSA_KEYS:
        mean_key = f'{key}_mean'
        if mean_key in clean_record:
            metrics[f'{key}_clean'] = clean_record[mean_key]
            metrics[f'{key}_corrupt_avg'] = float(np.mean([r.get(mean_key, 0.0) for r in all_corrupt])) if all_corrupt else 0.0
    for keys in (TEACHER_DIAG_KEYS, MEMORY_DIAG_KEYS):
        for key in keys:
            mean_key = f'{key}_mean'
            if mean_key in clean_record:
                metrics[f'{key}_clean'] = clean_record[mean_key]
                metrics[f'{key}_corrupt_avg'] = float(np.mean([r.get(mean_key, 0.0) for r in all_corrupt])) if all_corrupt else 0.0
    for key in ('hard_usage_delta_vs_src', 'dead_code_fraction_delta_vs_src'):
        if key in clean_record:
            metrics[f'{key}_clean'] = clean_record[key]
            metrics[f'{key}_corrupt_avg'] = float(np.mean([r.get(key, 0.0) for r in all_corrupt])) if all_corrupt else 0.0
    return metrics


def imagenet_c_curve_rows(variant_name, label, clean_record, corruption_records):
    rows = []
    clean_row = {'variant': variant_name, 'model': label, 'corruption': 'clean', 'severity': 0}
    clean_row.update(scalar_values(clean_record))
    rows.append(clean_row)
    for corr, records in corruption_records.items():
        for severity, record in zip(SEVERITIES, records):
            row = {'variant': variant_name, 'model': label, 'corruption': corr, 'severity': severity}
            row.update(scalar_values(record))
            rows.append(row)
    return rows


def write_markdown_table(rows, path):
    path = Path(path)
    ensure_dir(path.parent)
    if not rows:
        path.write_text('', encoding='utf-8')
        return
    df = pd.DataFrame(rows)
    cols = list(df.columns)
    header = '| ' + ' | '.join(str(c) for c in cols) + ' |'
    sep = '| ' + ' | '.join('---' for _ in cols) + ' |'
    body = []
    for _, row in df.iterrows():
        cells = []
        for c in cols:
            v = row[c]
            cells.append(f'{v:.4f}' if isinstance(v, float) else str(v))
        body.append('| ' + ' | '.join(cells) + ' |')
    path.write_text('\n'.join([header, sep, *body]), encoding='utf-8')


## Phase 0 — Pre-flight checks (gates duros antes del eval suite)

Cualquier gate fallido restringe o aborta variantes. Resultados se
persisten a `out/e10_phase0.json` para auditoría.


In [5]:
# Verify the ImageNet DeMemte checkpoint exists before Phase 0.
missing = [b for b in BASES if not ckpt_for_base(b).exists()]
if missing:
    raise FileNotFoundError(
        f'Falta checkpoint DeMemte ImageNet: {missing}. '
        f'Correr notebooks/06b_imagenet_train/e6_imagenet_train.ipynb primero.'
    )
print('checkpoint OK:', {b: str(ckpt_for_base(b)) for b in BASES})


checkpoint OK: {'dememte_imagenet_resnet50_vqsa': '/home/nakato/projects/Dememte/experiments/imagenet_dememte/out/dememte_imagenet_resnet50_vqsa_best.pt'}


In [6]:
# ---------------- P0.1 — Audit del gate de familiaridad ----------------
# Calibrar sigma para mediana(g_clean) en [0.3, 0.7] sobre ImageNet limpio,
# y reportar mediana(g) en condiciones ImageNet-C reales.

@torch.no_grad()
def collect_min_dists(model, loader, codebook, max_batches=8):
    model.eval()
    rows = []
    for i, (x, _) in enumerate(loader):
        if i >= max_batches:
            break
        x = x.to(device)
        feats = model.backbone(x)
        z = model.vqsa.projector(feats)
        z_pool = model.vqsa.pool(z).flatten(1)
        d2 = (z_pool.unsqueeze(1) - codebook.unsqueeze(0)).pow(2).sum(-1)
        rows.append(d2.min(dim=-1).values.detach().cpu())
    return torch.cat(rows) if rows else torch.empty(0)


phase0 = {}
sigma_by_base = {}

noise_loader, _ = imagenet_c_loader('gaussian_noise', 3)
pixel_loader, _ = imagenet_c_loader('pixelate', 3)

for base in BASES:
    model = load_base_model(base).eval()
    model.requires_grad_(False)
    cb = effective_codebook(model.vq)
    if cb is None:
        print(f'[{base}] no codebook (FSQ?), skipping P0.1')
        continue
    d_clean = collect_min_dists(model, clean_val_loader, cb)
    d_noise = collect_min_dists(model, noise_loader, cb)
    d_pixel = collect_min_dists(model, pixel_loader, cb)

    # Calibrate sigma so median(g_clean) ≈ 0.5.
    med_d_clean = float(d_clean.median().item())
    sigma2 = med_d_clean / math.log(2.0)
    sigma = math.sqrt(max(sigma2, 1e-8))
    sigma_by_base[base] = sigma

    def g_of(d2):
        return torch.exp(-d2 / (sigma ** 2))

    g_clean = g_of(d_clean)
    g_noise = g_of(d_noise)
    g_pixel = g_of(d_pixel)

    med_g_clean = float(g_clean.median().item())
    med_g_noise = float(g_noise.median().item())
    med_g_pixel = float(g_pixel.median().item())
    med_g_corrupt = min(med_g_noise, med_g_pixel)

    if med_g_corrupt > 0.05:
        gate_decision = 'familiarity viable'
    elif (1.0 - med_g_corrupt) > 0.95:
        gate_decision = 'familiarity inert in corrupt — use unfamiliarity or const'
    else:
        gate_decision = 'ambiguous — fallback to const gate'

    phase0[f'P0.1::{base}'] = dict(
        sigma=sigma,
        median_min_dist_clean=med_d_clean,
        median_g_clean=med_g_clean,
        median_g_gaussian_noise_s3=med_g_noise,
        median_g_pixelate_s3=med_g_pixel,
        gate_decision=gate_decision,
    )
    print(f'[{base}] sigma={sigma:.4f}  g_clean={med_g_clean:.3f}  '
          f'g_noise_s3={med_g_noise:.3f}  g_pixel_s3={med_g_pixel:.3f}  -> {gate_decision}')


[dememte_imagenet_resnet50_vqsa] sigma=3.8690  g_clean=0.500  g_noise_s3=0.463  g_pixel_s3=0.465  -> familiarity viable


In [7]:
# ---------------- P0.2 — Audit del codebook (Hopfield capacity) -------
# hard_usage < 10% ⇒ restringir esa base a variantes episodic_only.

HARD_USAGE_THRESHOLD = 0.10

@torch.no_grad()
def hard_usage_on_clean(model, loader, max_batches=20):
    model.eval()
    counts = None
    K = 0
    for i, (x, _) in enumerate(loader):
        if i >= max_batches:
            break
        x = x.to(device)
        _, _, dbg = model(x, return_debug=True)
        idx = dbg.get('encoding_indices')
        K = int(dbg.get('num_embeddings', 0) or 0)
        if idx is None or K == 0:
            return None
        bc = torch.bincount(idx.reshape(-1).long().cpu(), minlength=K).float()
        counts = bc if counts is None else counts + bc
    if counts is None or counts.sum().item() == 0:
        return None
    used = (counts > 0).float().mean().item()
    return float(used), K


base_restrictions = {}
for base in BASES:
    model = load_base_model(base).eval()
    result = hard_usage_on_clean(model, clean_val_loader)
    if result is None:
        base_restrictions[base] = 'episodic_only_lookup_free'
        phase0[f'P0.2::{base}'] = dict(hard_usage=None, decision='lookup_free → episodic_only')
        print(f'[{base}] lookup-free quantizer → restrict to episodic_only')
        continue
    usage, K = result
    if usage < HARD_USAGE_THRESHOLD:
        base_restrictions[base] = 'episodic_only_low_usage'
        decision = f'hard_usage={usage:.3f} < {HARD_USAGE_THRESHOLD} → episodic_only'
    else:
        base_restrictions[base] = 'all_variants'
        decision = f'hard_usage={usage:.3f} ≥ {HARD_USAGE_THRESHOLD} → all variants'
    phase0[f'P0.2::{base}'] = dict(hard_usage=usage, num_embeddings=K, decision=decision)
    print(f'[{base}] {decision}')


[dememte_imagenet_resnet50_vqsa] hard_usage=0.722 ≥ 0.1 → all variants


In [8]:
# ---------------- P0.3 — Clean accuracy floor con assoc_recall_const ---
# Catastrophic-forgetting gate: clean_acc(adapter, lambda=0.1) debe estar
# a ≤ 0.5 pp por debajo de source.clean_acc.

@torch.no_grad()
def clean_acc_of_adapter(adapter, loader):
    adapter.model.eval()
    total = correct = 0
    for x, y in loader:
        x = x.to(device); y = y.to(device)
        logits = adapter(x)
        pred = logits.argmax(1)
        total += y.size(0)
        correct += (pred == y).sum().item()
    return correct / max(1, total)


@torch.no_grad()
def clean_acc_of_model(model, loader):
    model.eval()
    total = correct = 0
    for x, y in loader:
        x = x.to(device); y = y.to(device)
        pred = model(x).argmax(1)
        total += y.size(0)
        correct += (pred == y).sum().item()
    return correct / max(1, total)


p03_results = {}
for base in BASES:
    sigma = sigma_by_base.get(base, 1.0)
    model = load_base_model(base)
    src_acc = clean_acc_of_model(model, clean_val_loader)

    if base_restrictions[base] == 'all_variants':
        cfg = HippocampalConfig(
            recall_sem=True, recall_epi=False, T=1,
            gate_mode='const', lambda_max=LAMBDA_MAX, tau=TAU, sigma=sigma,
        )
        adapter = HippocampalMemoryAdapter(load_base_model(base), cfg)
        adapter_acc = clean_acc_of_adapter(adapter, clean_val_loader)
    else:
        cfg = HippocampalConfig(
            recall_sem=False, recall_epi=True, T=1,
            gate_mode='const', lambda_max=LAMBDA_MAX, tau=TAU,
            tau_epi=TAU_EPI, sigma=sigma,
        )
        adapter = HippocampalMemoryAdapter(load_base_model(base), cfg)
        adapter_acc = clean_acc_of_adapter(adapter, clean_val_loader)

    delta = adapter_acc - src_acc
    passes = delta >= -CLEAN_FLOOR_TOL
    p03_results[base] = dict(source_clean=src_acc, adapter_clean=adapter_acc, delta=delta, passes=bool(passes))
    phase0[f'P0.3::{base}'] = p03_results[base]
    print(f'[{base}] src_clean={src_acc:.4f}  adapter_clean={adapter_acc:.4f}  '
          f'delta={delta:+.4f}  passes_floor={passes}')

write_json(phase0, OUT / 'e10_phase0.json')
print('Phase 0 results written to', OUT / 'e10_phase0.json')

bases_to_skip = [b for b, r in p03_results.items() if not r['passes']]
if bases_to_skip:
    print(f'WARNING: bases failing clean-acc floor → skipped: {bases_to_skip}')


[dememte_imagenet_resnet50_vqsa] src_clean=0.6140  adapter_clean=0.6160  delta=+0.0020  passes_floor=True
Phase 0 results written to /home/nakato/projects/Dememte/notebooks/10_memory_hippocampal/out/e10_phase0.json


## Definición de variantes (post-Phase 0)


In [9]:
def variants_for_base(base):
    # Return list of (variant_name, HippocampalConfig) tuples for a base.
    sigma = sigma_by_base.get(base, 1.0)
    restr = base_restrictions[base]

    # Decide gate mode per base from P0.1 decision.
    decision = phase0.get(f'P0.1::{base}', {}).get('gate_decision', '')
    if 'familiarity viable' in decision:
        best_gate = 'familiarity'
    elif 'unfamiliarity' in decision:
        best_gate = 'unfamiliarity'
    else:
        best_gate = 'const'

    base_kwargs = dict(
        lambda_max=LAMBDA_MAX, tau=TAU, tau_epi=TAU_EPI, sigma=sigma,
        alpha_w=ALPHA_W, episodic_size=EPI_SIZE,
    )

    out = []

    # episodic_only is always valid (no semantic dependence).
    out.append((
        'episodic_only',
        HippocampalConfig(
            recall_sem=False, recall_epi=True, T=1, gate_mode='const',
            beta=0.0, episodic_init_from_codebook=(restr == 'all_variants'),
            **base_kwargs,
        ),
    ))

    if restr != 'all_variants':
        # P0.2 restricted this base; only episodic_only is reportable.
        return out

    # Semantic-dependent variants.
    out += [
        ('assoc_recall_const', HippocampalConfig(
            recall_sem=True, recall_epi=False, T=1, gate_mode='const',
            beta=1.0, **base_kwargs)),
        ('assoc_recall_familiarity', HippocampalConfig(
            recall_sem=True, recall_epi=False, T=1, gate_mode='familiarity',
            beta=1.0, **base_kwargs)),
        ('assoc_recall_unfamiliarity', HippocampalConfig(
            recall_sem=True, recall_epi=False, T=1, gate_mode='unfamiliarity',
            beta=1.0, **base_kwargs)),
        ('completion_T3_best_gate', HippocampalConfig(
            recall_sem=True, recall_epi=False, T=3, gate_mode=best_gate,
            beta=1.0, **base_kwargs)),
        ('hippocampal_full', HippocampalConfig(
            recall_sem=True, recall_epi=True, T=3, gate_mode=best_gate,
            beta=BETA_MIX, episodic_init_from_codebook=True, **base_kwargs)),
        ('consolidation_slow', HippocampalConfig(
            recall_sem=True, recall_epi=True, T=3, gate_mode=best_gate,
            beta=BETA_MIX, episodic_init_from_codebook=True,
            alpha_s=ALPHA_S, consolidation_every=50, **base_kwargs)),
    ]
    return out


## Run E10 — eval suite por base × variante


In [10]:
all_summaries = []
all_curves = []

for base in BASES:
    if base in bases_to_skip:
        print(f'=== {base} skipped (P0.3 failed) ===')
        continue
    print(f'=== BASE: {base} ===')
    teacher = load_base_model(base).eval()
    teacher.requires_grad_(False)
    cfg = cfg_for_base(base)
    ckpt = ckpt_for_base(base)

    # Source baseline on clean ImageNet and real ImageNet-C condition loaders.
    source_model = load_base_model(base)
    source_clean = evaluate_dememte(source_model, clean_val_loader, device=device)
    source_corrupt = {}
    for corruption in CORRUPTIONS:
        source_corrupt[corruption] = []
        for severity in SEVERITIES:
            loader, cmeta = imagenet_c_loader(corruption, severity)
            print(f'  source :: {corruption}/{severity} ({cmeta["size"]} samples)')
            source_corrupt[corruption].append(evaluate_dememte(source_model, loader, device=device))
    src_metrics = summarize_imagenet_c_records(source_clean, source_corrupt)

    for variant_name, hp_cfg in variants_for_base(base):
        print(f'  -- {base} :: {variant_name}')

        def factory(hp_cfg=hp_cfg):
            model = load_base_model(base)
            return HippocampalMemoryAdapter(model, hp_cfg)

        clean_record = evaluate_dememte_tta(
            factory(),
            clean_val_loader,
            device=device,
            tta_method=variant_name,
            tta_base_variant=base,
            teacher_model=teacher,
        )
        corrupt_records = {}
        for corruption in CORRUPTIONS:
            corrupt_records[corruption] = []
            for severity in SEVERITIES:
                loader, cmeta = imagenet_c_loader(corruption, severity)
                print(f'     {corruption}/{severity} ({cmeta["size"]} samples)')
                corrupt_records[corruption].append(evaluate_dememte_tta(
                    factory(),
                    loader,
                    device=device,
                    tta_method=variant_name,
                    tta_base_variant=base,
                    teacher_model=teacher,
                ))
        metrics = summarize_imagenet_c_records(clean_record, corrupt_records)
        label = f'{base}::{variant_name}'
        curve_rows = imagenet_c_curve_rows(variant_name, label, clean_record, corrupt_records)

        summary = {k: v for k, v in metrics.items()
                   if isinstance(v, (int, float, bool, str, np.floating))}
        summary.update({
            'variant': variant_name,
            'label': label,
            'base_variant': base,
            'base_checkpoint': str(ckpt),
            'dataset': meta['dataset'],
            'clean_root': str(DATA_ROOT_CLEAN),
            'imagenet_c_root': str(DATA_ROOT_C),
            'corruptions': ','.join(CORRUPTIONS),
            'severities': ','.join(str(s) for s in SEVERITIES),
            'max_samples_per_class': MAX_SAMPLES_PER_CLASS,
            'quantizer_type': cfg.quantizer_type,
            'delta_clean_vs_source': metrics['clean_acc'] - src_metrics['clean_acc'],
            'delta_corrupt_vs_source': metrics['corrupt_acc_avg'] - src_metrics['corrupt_acc_avg'],
            'passes_clean_floor': bool(metrics['clean_acc'] >= src_metrics['clean_acc'] - CLEAN_FLOOR_TOL),
        })
        all_summaries.append(summary)
        all_curves.extend(curve_rows)

        method_dir = ensure_dir(OUT / base / variant_name)
        write_json(summary, method_dir / 'metrics.json')
        write_csv(curve_rows, method_dir / 'signal_curves.csv')

        report_keys = ['clean_acc', 'corrupt_acc_avg', 'delta_clean_vs_source',
                       'delta_corrupt_vs_source', 'completion_amount_corrupt_avg',
                       'recall_sharpness_corrupt_avg', 'g_mean_corrupt_avg']
        print('     ', {k: round(float(summary[k]), 4) for k in report_keys if k in summary})

    src_row = {k: v for k, v in src_metrics.items()
               if isinstance(v, (int, float, bool, str, np.floating))}
    src_row.update({
        'variant': 'source', 'label': f'{base}::source', 'base_variant': base,
        'base_checkpoint': str(ckpt), 'dataset': meta['dataset'],
        'clean_root': str(DATA_ROOT_CLEAN), 'imagenet_c_root': str(DATA_ROOT_C),
        'corruptions': ','.join(CORRUPTIONS),
        'severities': ','.join(str(s) for s in SEVERITIES),
        'max_samples_per_class': MAX_SAMPLES_PER_CLASS,
        'quantizer_type': cfg.quantizer_type,
        'delta_clean_vs_source': 0.0, 'delta_corrupt_vs_source': 0.0,
        'passes_clean_floor': True,
    })
    all_summaries.append(src_row)
    all_curves.extend(imagenet_c_curve_rows('source', f'{base}::source', source_clean, source_corrupt))

write_csv(all_summaries, OUT / 'e10_results.csv')
write_csv(all_curves, OUT / 'e10_curves.csv')

ranked = sorted(all_summaries, key=lambda r: r.get('corrupt_acc_avg', 0.0), reverse=True)
write_markdown_table(ranked, OUT / 'e10_summary.md')
pd.DataFrame(ranked).head(20)


=== BASE: dememte_imagenet_resnet50_vqsa ===
  source :: gaussian_noise/3 (50000 samples)
  source :: gaussian_noise/5 (50000 samples)
  source :: motion_blur/3 (50000 samples)
  source :: motion_blur/5 (50000 samples)
  source :: pixelate/3 (50000 samples)
  source :: pixelate/5 (50000 samples)
  source :: jpeg_compression/3 (50000 samples)
  source :: jpeg_compression/5 (50000 samples)
  -- dememte_imagenet_resnet50_vqsa :: episodic_only
     gaussian_noise/3 (50000 samples)
     gaussian_noise/5 (50000 samples)
     motion_blur/3 (50000 samples)
     motion_blur/5 (50000 samples)
     pixelate/3 (50000 samples)
     pixelate/5 (50000 samples)
     jpeg_compression/3 (50000 samples)
     jpeg_compression/5 (50000 samples)
      {'clean_acc': 0.614, 'corrupt_acc_avg': 0.2171, 'delta_clean_vs_source': 0.0, 'delta_corrupt_vs_source': -0.0003, 'completion_amount_corrupt_avg': 0.0462, 'recall_sharpness_corrupt_avg': 0.5299, 'g_mean_corrupt_avg': 1.0}
  -- dememte_imagenet_resnet50_vqsa ::

,clean_acc,corrupt_acc_avg,corrupt_acc_gaussian_noise,corrupt_acc_motion_blur,corrupt_acc_pixelate,corrupt_acc_jpeg_compression,ece_clean,ece_corrupt_avg,nll_clean,nll_corrupt_avg,...,dataset,clean_root,imagenet_c_root,corruptions,severities,max_samples_per_class,quantizer_type,delta_clean_vs_source,delta_corrupt_vs_source,passes_clean_floor
0,0.616,0.217463,0.14554,0.13246,0.24080,0.35105,0.130219,0.147806,2.048646,5.122316,...,imagenet_c,/home/nakato/projects/Dememte/experiments/data...,/home/nakato/projects/Dememte/experiments/data...,"gaussian_noise,motion_blur,pixelate,jpeg_compr...","3,5",None,ema_vq,0.002,0.000060,True
1,0.616,0.217440,0.14552,0.13259,0.24081,0.35084,0.130048,0.147300,2.048701,5.116580,...,imagenet_c,/home/nakato/projects/Dememte/experiments/data...,/home/nakato/projects/Dememte/experiments/data...,"gaussian_noise,motion_blur,pixelate,jpeg_compr...","3,5",None,ema_vq,0.002,0.000038,True
2,0.617,0.217427,0.14550,0.13262,0.24082,0.35077,0.129002,0.147338,2.048749,5.116065,...,imagenet_c,/home/nakato/projects/Dememte/experiments/data...,/home/nakato/projects/Dememte/experiments/data...,"gaussian_noise,motion_blur,pixelate,jpeg_compr...","3,5",None,ema_vq,0.003,0.000025,True
3,0.616,0.217417,0.14547,0.13260,0.24081,0.35079,0.129977,0.147266,2.048784,5.115316,...,imagenet_c,/home/nakato/projects/Dememte/experiments/data...,/home/nakato/projects/Dememte/experiments/data...,"gaussian_noise,motion_blur,pixelate,jpeg_compr...","3,5",None,ema_vq,0.002,0.000015,True
4,0.614,0.217402,0.14533,0.13268,0.24084,0.35076,0.131667,0.146770,2.048999,5.110213,...,imagenet_c,/home/nakato/projects/Dememte/experiments/data...,/home/nakato/projects/Dememte/experiments/data...,"gaussian_noise,motion_blur,pixelate,jpeg_compr...","3,5",None,ema_vq,0.000,0.000000,True
5,0.617,0.217310,0.14535,0.13244,0.24070,0.35075,0.128916,0.146599,2.049181,5.112487,...,imagenet_c,/home/nakato/projects/Dememte/experiments/data...,/home/nakato/projects/Dememte/experiments/data...,"gaussian_noise,motion_blur,pixelate,jpeg_compr...","3,5",None,ema_vq,0.003,-0.000092,True
6,0.617,0.217308,0.14538,0.13243,0.24064,0.35078,0.128976,0.146604,2.049001,5.112429,...,imagenet_c,/home/nakato/projects/Dememte/experiments/data...,/home/nakato/projects/Dememte/experiments/data...,"gaussian_noise,motion_blur,pixelate,jpeg_compr...","3,5",None,ema_vq,0.003,-0.000095,True
7,0.614,0.217107,0.14525,0.13173,0.24060,0.35085,0.131977,0.144914,2.048245,5.108294,...,imagenet_c,/home/nakato/projects/Dememte/experiments/data...,/home/nakato/projects/Dememte/experiments/data...,"gaussian_noise,motion_blur,pixelate,jpeg_compr...","3,5",None,ema_vq,0.000,-0.000295,True


## Lectura

**Hito mecánico (criterio principal, no accuracy).** Antes de leer
accuracy, verificar:

1. `completion_amount_corrupt_avg > 0` para variantes con `T ≥ 1` →
   confirma que la inyección estructural muerde (Ramsauer 2021 Eq. 7).
2. `episodic_buffer_churn_corrupt_avg > 0` para variantes con
   `recall_epi=True` → confirma que el buffer episódico se escribe
   (Sun-Saxe-Fitzgerald 2023 plasticidad rápida).
3. `traj_max_step_corrupt_avg` finito y no creciente con `T` → confirma
   que el loop converge (Kim 2021).

**Floor de clean accuracy (gate duro).** `passes_clean_floor=True` para
cada variante reportada en el ranking final. Si una variante regresiona
clean acc, se reporta separadamente (Wang 2022 CoTTA, Song 2023 EcoTTA).

**Comparativa de aislamiento.** `delta_corrupt_vs_source` cuantifica la
ganancia neta; comparar con `assoc_recall_const` para aislar si la
ganancia viene del gate biológico / iteración / episódico, o sólo de
"softear" el `argmin` del VQ.
